# LAB 02 [SOLUTION] — ELT Pipeline: Ingestion → Bronze → Silver

**Databricks Fundamentals | ~75 min | Difficulty **

---

## Scenario

You are a data engineer at **RetailHub**, a mid-size e-commerce company.
Three raw data files have landed on a shared Unity Catalog Volume:

| File | Format | Contents |
|------|--------|----------|
| `customers/customers.csv` | CSV | ~1 000 customer records |
| `orders/orders_batch.json` | JSON | ~5 000 order records |
| `products/products.csv` | CSV | product catalog |

Your task: build a **Bronze → Silver ELT pipeline** that:
1. Reads each file with an **explicit schema** (no inferSchema in production)
2. Adds audit metadata (`_load_ts`, `_source_file`)
3. Cleans, casts, enriches and joins the data
4. Writes typed Silver Delta tables
5. Creates a reporting view for the analytics team

## Task map

| # | Title | Type | Difficulty |
|---|-------|------|-----------|
| 01 | Explore the Volume | GIVEN |  |
| 02 | Read CSV — spot schema problems | GIVEN |  |
| 03 | Define StructType for customers | **TODO** |  |
| 04 | Read JSON orders with explicit schema | **TODO** |  |
| 05 | Add audit metadata columns | **TODO** |  |
| 06 | Write Bronze Delta tables | **TODO** |  |
| 07 | Cast & clean Silver customers | **TODO** |  |
| 08 | Enrich Silver orders | **TODO** |  |
| 09 | Join orders ↔ customers | **TODO** |  |
| 10 | Aggregate: revenue per customer | **TODO** |  |
| 11 | Write Silver Delta table (partitioned) | **TODO** |  |
| 12 | Temp view + ad-hoc SQL analysis | **TODO** |  |

In [ ]:
# Setup — run this first
%run ../setup/00_setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 01 — Explore the Volume                                      [GIVEN]
# ─────────────────────────────────────────────────────────────────────────
# dbutils.fs.ls() lists files on a DBFS/Volume path.

print("=== datasets root ===")
for f in dbutils.fs.ls(DATASET_PATH):
    print(f"  {f.name:40s}  {f.size:>10,} bytes")

print("\n=== orders/ ===")
for f in dbutils.fs.ls(f"{DATASET_PATH}/orders"):
    print(f"  {f.name:40s}  {f.size:>10,} bytes")

# ? How many files are in orders/?
# ? Is the file size of orders_batch.json reasonable for ~5 000 rows?

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 02 — Read customers.csv: auto-schema, spot the problems      [GIVEN]
# ─────────────────────────────────────────────────────────────────────────

df_infer = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(f"{DATASET_PATH}/customers/customers.csv")
)

print("=== Schema inferred by Spark ===")
df_infer.printSchema()
df_infer.show(5)

# ? What type did Spark assign to registration_date? Is it correct?
# ? What type is loyalty_points? What would happen if a null slips in?
# ? Why is inferSchema risky in a production pipeline?

### Task 03 — Define StructType for customers  `[TODO]`

The inferred schema above uses **StringType** for dates and may guess
numeric types incorrectly. Define a manual `StructType` and re-read the file.

**API hint — constructing a schema:**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("column_name", DataType(), nullable=True),
    ...
])

df = (spark.read
           .schema(schema)
           .option("header", "true")
           .option("dateFormat", "yyyy-MM-dd")
           .csv(path))
```

**Customers CSV columns:**
`customer_id` (String, NOT NULL), `first_name` (String), `last_name` (String),
`email` (String), `country` (String), `registration_date` (**DateType**),
`loyalty_points` (**IntegerType**), `annual_spend` (**DoubleType**)

In [ ]:
# Task 03 [SOLUTION]
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,DateType

customer_schema = StructType([
    StructField('customer_id',       StringType(),  False),
    StructField('first_name',        StringType(),  True),
    StructField('last_name',         StringType(),  True),
    StructField('email',             StringType(),  True),
    StructField('country',           StringType(),  True),
    StructField('registration_date', DateType(),    True),
    StructField('loyalty_points',    IntegerType(), True),
    StructField('annual_spend',      DoubleType(),  True),
])

df_customers_raw = (spark.read
    .schema(customer_schema)
    .option('header','true')
    .option('dateFormat','yyyy-MM-dd')
    .csv(f'{DATASET_PATH}/customers/customers.csv'))

df_customers_raw.printSchema()
print(f'Row count: {df_customers_raw.count()}')
df_customers_raw.show(5)

assert dict(df_customers_raw.dtypes)['registration_date'] == 'date'
assert dict(df_customers_raw.dtypes)['loyalty_points'] == 'int'
print('✓ Task 03 passed')

### Task 04 — Read JSON orders with explicit schema  `[TODO]`

`inferSchema` on JSON reads the **entire file** to detect types — expensive on large files.
A manual schema lets you control `TimestampType` for datetime fields.

**API hint — JSON with explicit schema:**
```python
from pyspark.sql.types import TimestampType

df = (
    spark.read
         .schema(orders_schema)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .json(path)
)
```

**Orders JSON fields:**
`order_id` (String), `customer_id` (String), `product_id` (String),
`store_id` (String), `order_datetime` (**TimestampType**),
`quantity` (IntegerType), `unit_price` (DoubleType),
`discount_percent` (DoubleType), `total_amount` (DoubleType),
`payment_method` (String), `status` (String)

In [ ]:
# Task 04 [SOLUTION]
from pyspark.sql.types import TimestampType

orders_schema = StructType([
    StructField('order_id',         StringType(),    False),
    StructField('customer_id',      StringType(),    True),
    StructField('product_id',       StringType(),    True),
    StructField('store_id',         StringType(),    True),
    StructField('order_datetime',   TimestampType(), True),
    StructField('quantity',         IntegerType(),   True),
    StructField('unit_price',       DoubleType(),    True),
    StructField('discount_percent', DoubleType(),    True),
    StructField('total_amount',     DoubleType(),    True),
    StructField('payment_method',   StringType(),    True),
    StructField('status',           StringType(),    True),
])

df_orders_raw = (spark.read
    .schema(orders_schema)
    .option('timestampFormat','yyyy-MM-dd HH:mm:ss')
    .json(f'{DATASET_PATH}/orders/orders_batch.json'))

df_orders_raw.printSchema()
print(f'Row count: {df_orders_raw.count()}')
df_orders_raw.show(5,truncate=False)

assert dict(df_orders_raw.dtypes)['order_datetime'] == 'timestamp'
assert dict(df_orders_raw.dtypes)['total_amount'] == 'double'
print('✓ Task 04 passed')

### Task 05 — Add audit metadata columns  `[TODO]`

Every Bronze table must record **when** and **from where** each row was loaded.
This is critical for debugging incremental loads.

**API hint:**
```python
from pyspark.sql import functions as F

df = (
    df
    .withColumn("_load_ts",     F.current_timestamp())   # TimestampType
    .withColumn("_source_file", F.lit("path/to/file"))   # StringType literal
)
```

In [ ]:
# Task 05 [SOLUTION]
from pyspark.sql import functions as F

CUSTOMERS_PATH = f'{DATASET_PATH}/customers/customers.csv'
ORDERS_PATH    = f'{DATASET_PATH}/orders/orders_batch.json'

df_customers_bronze = (df_customers_raw
    .withColumn('_load_ts',     F.current_timestamp())
    .withColumn('_source_file', F.lit(CUSTOMERS_PATH)))

df_orders_bronze = (df_orders_raw
    .withColumn('_load_ts',     F.current_timestamp())
    .withColumn('_source_file', F.lit(ORDERS_PATH)))

assert '_load_ts'     in df_customers_bronze.columns
assert '_source_file' in df_orders_bronze.columns
print('✓ Task 05 passed')

### Task 06 — Write Bronze Delta tables  `[TODO]`

Write both enriched DataFrames to managed Delta tables in the `bronze` schema.
Use **overwrite** mode — Bronze is always a full reload from the source file.

**API hint — writing a managed Delta table (Unity Catalog):**
```python
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.table_name")
)
```

Target tables:
- `{CATALOG}.bronze.lab_customers`
- `{CATALOG}.bronze.lab_orders`

In [ ]:
# Task 06 [SOLUTION]
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze')

(df_customers_bronze.write.format('delta').mode('overwrite')
    .option('overwriteSchema','true')
    .saveAsTable(f'{CATALOG}.bronze.lab_customers'))

(df_orders_bronze.write.format('delta').mode('overwrite')
    .option('overwriteSchema','true')
    .saveAsTable(f'{CATALOG}.bronze.lab_orders'))

print(f'bronze.lab_customers: {spark.table(f"{CATALOG}.bronze.lab_customers").count():,}')
print(f'bronze.lab_orders:    {spark.table(f"{CATALOG}.bronze.lab_orders").count():,}')

In [ ]:
#  Check — Task 06: Bronze tables written
_cust = spark.table(f"{CATALOG}.bronze.lab_customers")
_ord  = spark.table(f"{CATALOG}.bronze.lab_orders")

assert _cust.count() > 0, "bronze.lab_customers is empty — did you write the DataFrame?"
assert _ord.count() > 0,  "bronze.lab_orders is empty — did you write the DataFrame?"
assert "_load_ts"     in _cust.columns, "_load_ts column missing from bronze.lab_customers"
assert "_source_file" in _ord.columns,  "_source_file column missing from bronze.lab_orders"
print(f" Task 06 passed — bronze.lab_customers: {_cust.count():,} rows | bronze.lab_orders: {_ord.count():,} rows")

### Task 07 — Cast & clean Silver customers  `[TODO]`

Silver is **typed, clean and enriched**. Apply the following rules:

1. Keep only: `customer_id`, `first_name`, `last_name`, `email`, `country`,
   `registration_date`, `loyalty_points`, `annual_spend`
2. Drop rows where `customer_id` IS NULL or empty string
3. Normalise `country` to UPPER CASE
4. Add `full_name = first_name + " " + last_name`
5. Add `loyalty_tier`:
   - `"gold"` if `loyalty_points >= 1000`
   - `"silver"` if `loyalty_points >= 500`
   - `"bronze"` otherwise

**API hints:**
```python
# conditional column
F.when(condition, value).when(condition2, value2).otherwise(default)

# string concat
F.concat_ws(" ", F.col("first_name"), F.col("last_name"))

# null / empty guard
F.col("x").isNotNull() & (F.col("x") != "")

# upper case
F.upper(F.col("country"))
```

In [ ]:
# Task 07 [SOLUTION]
from pyspark.sql import functions as F

df_silver_customers = (spark.table(f'{CATALOG}.bronze.lab_customers')
    .select('customer_id','first_name','last_name','email','country',
            'registration_date','loyalty_points','annual_spend')
    .filter(F.col('customer_id').isNotNull() & (F.col('customer_id') != ''))
    .withColumn('country',      F.upper(F.col('country')))
    .withColumn('full_name',    F.concat_ws(' ', F.col('first_name'), F.col('last_name')))
    .withColumn('loyalty_tier', F.when(F.col('loyalty_points')>=1000,'gold')
                                 .when(F.col('loyalty_points')>=500, 'silver')
                                 .otherwise('bronze')))

df_silver_customers.show(5,truncate=False)
df_silver_customers.groupBy('loyalty_tier').count().show()
assert 'full_name'    in df_silver_customers.columns
assert 'loyalty_tier' in df_silver_customers.columns
print(f'✓ Task 07 passed ({df_silver_customers.count():,} rows)')

In [ ]:
#  Check — Task 07: Silver customers
assert df_silver_customers is not None, "df_silver_customers is None"
assert df_silver_customers.count() > 0, "df_silver_customers is empty"
assert "full_name"    in df_silver_customers.columns, "Missing column: full_name"
assert "loyalty_tier" in df_silver_customers.columns, "Missing column: loyalty_tier"
assert "country"      in df_silver_customers.columns, "Missing column: country"

# No nulls in customer_id (must have been filtered)
null_ids = df_silver_customers.filter("customer_id IS NULL OR customer_id = ''").count()
assert null_ids == 0, f"Found {null_ids} rows with null/empty customer_id — filter not applied?"

# country must be upper case
from pyspark.sql import functions as F
non_upper = df_silver_customers.filter(F.col("country") != F.upper(F.col("country"))).count()
assert non_upper == 0, f"{non_upper} rows have non-upper-case country values"

# loyalty_tier values must be valid
valid_tiers = {"gold", "silver", "bronze"}
actual_tiers = {r.loyalty_tier for r in df_silver_customers.select("loyalty_tier").distinct().collect()}
assert actual_tiers.issubset(valid_tiers), f"Unexpected loyalty_tier values: {actual_tiers - valid_tiers}"

print(f" Task 07 passed — {df_silver_customers.count():,} clean customer rows | tiers: {actual_tiers}")

### Task 08 — Enrich Silver orders  `[TODO]`

Business rules for `df_silver_orders`:

1. Keep only orders with `status` IN (`'shipped'`, `'delivered'`, `'returned'`)
2. Extract `order_date` (DateType) and `order_hour` (IntegerType) from `order_datetime`
3. Calculate `revenue_net = total_amount * (1 - discount_percent / 100.0)`
4. Add boolean `is_discounted = discount_percent > 0`

**API hints:**
```python
F.col("status").isin(["shipped", "delivered", "returned"])
F.to_date(F.col("order_datetime").cast("timestamp"))
F.hour(F.col("order_datetime").cast("timestamp"))
F.col("discount_percent") > 0
```

In [ ]:
# Task 08 [SOLUTION]
from pyspark.sql import functions as F

df_silver_orders = (spark.table(f'{CATALOG}.bronze.lab_orders')
    .filter(F.col('status').isin(['shipped','delivered','returned']))
    .withColumn('order_date',    F.to_date(F.col('order_datetime').cast('timestamp')))
    .withColumn('order_hour',    F.hour(F.col('order_datetime').cast('timestamp')))
    .withColumn('revenue_net',   F.round(F.col('total_amount')*(1-F.col('discount_percent')/100.0),2))
    .withColumn('is_discounted', F.col('discount_percent') > 0))

df_silver_orders.show(5,truncate=False)
df_silver_orders.groupBy('status').count().show()
for col in ['order_date','order_hour','revenue_net','is_discounted']:
    assert col in df_silver_orders.columns
print(f'✓ Task 08 passed ({df_silver_orders.count():,} rows)')

In [ ]:
#  Check — Task 08: Silver orders
assert df_silver_orders is not None, "df_silver_orders is None"
assert df_silver_orders.count() > 0, "df_silver_orders is empty"

for col in ["order_date", "order_hour", "revenue_net", "is_discounted"]:
    assert col in df_silver_orders.columns, f"Missing column: {col}"

# 'pending' and 'cancelled' must have been filtered out
from pyspark.sql import functions as F
bad_statuses = df_silver_orders.filter(F.col("status").isin(["pending", "cancelled"])).count()
assert bad_statuses == 0, f"Found {bad_statuses} rows with status 'pending' or 'cancelled' — filter not applied?"

# order_date should be DateType
assert dict(df_silver_orders.dtypes)["order_date"] == "date", "order_date must be DateType"
assert dict(df_silver_orders.dtypes)["order_hour"] == "int",  "order_hour must be IntegerType"

print(f" Task 08 passed — {df_silver_orders.count():,} clean order rows")

### Task 09 — Join orders ↔ customers  `[TODO]`

Produce a flat enriched DataFrame combining order and customer dimensions.

- Join `df_silver_orders` (left) with `df_silver_customers` (right) on `customer_id`
- Use a **left join** — preserve all orders even without a matching customer

**API hint:**
```python
df_joined = df_left.join(
    df_right,
    on  = "join_key",   # column name, list, or boolean
    how = "left"        # inner | left | right | full | semi | anti
)
```

Keep only:
`order_id`, `order_date`, `order_hour`, `status`, `revenue_net`,
`is_discounted`, `payment_method`, `customer_id`,
`full_name`, `country`, `loyalty_tier`

In [ ]:
# Task 09 [SOLUTION]
from pyspark.sql import functions as F

df_joined = (df_silver_orders
    .join(df_silver_customers, on='customer_id', how='left')
    .select('order_id','order_date','order_hour','status','revenue_net',
            'is_discounted','payment_method','customer_id',
            'full_name','country','loyalty_tier'))

df_joined.show(5,truncate=False)
assert df_joined.count() >= df_silver_orders.count()
print(f'✓ Task 09 passed ({df_joined.count():,} rows)')

In [ ]:
#  Check — Task 09: Orders ↔ Customers join
assert df_joined is not None, "df_joined is None"
assert df_joined.count() > 0, "df_joined is empty"

expected_cols = ["order_id", "order_date", "status", "revenue_net",
                 "customer_id", "full_name", "country", "loyalty_tier"]
for col in expected_cols:
    assert col in df_joined.columns, f"Missing column in df_joined: {col}"

# Left join — count must be >= silver_orders count
assert df_joined.count() >= df_silver_orders.count(), \
    "Left join shrunk the order count — did you use 'inner' instead of 'left'?"

print(f" Task 09 passed — {df_joined.count():,} joined rows | all expected columns present")

### Task 10 — Aggregate: revenue per customer  `[TODO]`

Produce a Gold-style customer summary.

| Output column | How to compute |
|---|---|
| `customer_id` | group key |
| `full_name` | group key |
| `country` | group key |
| `loyalty_tier` | group key |
| `total_orders` | `F.count("order_id")` |
| `total_revenue` | `F.sum("revenue_net")` rounded 2 dp |
| `avg_order_value` | `F.mean("revenue_net")` rounded 2 dp |
| `first_order_date` | `F.min("order_date")` |
| `last_order_date` | `F.max("order_date")` |

Sort by `total_revenue` descending.

**API hint:**
```python
df.groupBy("a","b").agg(
    F.count("x").alias("cnt"),
    F.round(F.sum("y"), 2).alias("total")
).orderBy(F.col("total").desc())
```

In [ ]:
# Task 10 [SOLUTION]
from pyspark.sql import functions as F

df_customer_summary = (df_joined
    .groupBy('customer_id','full_name','country','loyalty_tier')
    .agg(
        F.count('order_id').alias('total_orders'),
        F.round(F.sum('revenue_net'),  2).alias('total_revenue'),
        F.round(F.mean('revenue_net'), 2).alias('avg_order_value'),
        F.min('order_date').alias('first_order_date'),
        F.max('order_date').alias('last_order_date'),
    )
    .orderBy(F.col('total_revenue').desc()))

df_customer_summary.show(10,truncate=False)
print(f'✓ Task 10 passed ({df_customer_summary.count():,} customers)')

In [ ]:
#  Check — Task 10: Customer revenue summary
assert df_customer_summary is not None, "df_customer_summary is None"
assert df_customer_summary.count() > 0, "df_customer_summary is empty"

for col in ["customer_id", "full_name", "total_orders", "total_revenue",
            "avg_order_value", "first_order_date", "last_order_date"]:
    assert col in df_customer_summary.columns, f"Missing column: {col}"

# total_revenue must be positive
from pyspark.sql import functions as F
neg_revenue = df_customer_summary.filter(F.col("total_revenue") <= 0).count()
assert neg_revenue == 0, f"{neg_revenue} rows have non-positive total_revenue"

# one row per customer
total_rows = df_customer_summary.count()
distinct_customers = df_customer_summary.select("customer_id").distinct().count()
assert total_rows == distinct_customers, "Duplicate customer_id rows in summary — check groupBy keys"

print(f" Task 10 passed — {total_rows:,} unique customers in revenue summary")

### Task 11 — Write Silver Delta table (partitioned)  `[TODO]`

Write `df_silver_orders` to `{CATALOG}.silver.lab_orders` **partitioned by `order_date`**
so daily queries skip irrelevant partitions.

**API hint — partitioned write:**
```python
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("partition_column")
    .saveAsTable(f"{CATALOG}.silver.table_name")
)
```

Also write `df_silver_customers` to `{CATALOG}.silver.lab_customers`
(no partitioning).

In [ ]:
# Task 11 [SOLUTION]
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver')

(df_silver_orders.write.format('delta').mode('overwrite')
    .option('overwriteSchema','true')
    .partitionBy('order_date')
    .saveAsTable(f'{CATALOG}.silver.lab_orders'))

(df_silver_customers.write.format('delta').mode('overwrite')
    .option('overwriteSchema','true')
    .saveAsTable(f'{CATALOG}.silver.lab_customers'))

detail = spark.sql(f'DESCRIBE DETAIL {CATALOG}.silver.lab_orders').collect()[0]
assert 'order_date' in detail['partitionColumns']
print('✓ Task 11 passed — silver tables written')

In [ ]:
#  Check — Task 11: Silver tables written
_silver_orders = spark.table(f"{CATALOG}.silver.lab_orders")
_silver_cust   = spark.table(f"{CATALOG}.silver.lab_customers")

assert _silver_orders.count() > 0, "silver.lab_orders is empty — did you write the DataFrame?"
assert _silver_cust.count() > 0,   "silver.lab_customers is empty — did you write the DataFrame?"

# lab_orders must be partitioned by order_date
detail = spark.sql(f"DESCRIBE DETAIL {CATALOG}.silver.lab_orders").collect()[0]
assert "order_date" in detail["partitionColumns"], \
    f"silver.lab_orders is not partitioned by order_date. Got: {detail['partitionColumns']}"

# verify required Silver columns exist
for col in ["order_date", "order_hour", "revenue_net", "is_discounted"]:
    assert col in _silver_orders.columns, f"Missing column in silver.lab_orders: {col}"
for col in ["full_name", "loyalty_tier", "country"]:
    assert col in _silver_cust.columns, f"Missing column in silver.lab_customers: {col}"

print(f" Task 11 passed — silver.lab_orders: {_silver_orders.count():,} rows (partitioned by order_date)")
print(f"                    silver.lab_customers: {_silver_cust.count():,} rows")

### Task 12 — Temp view + ad-hoc SQL analysis  `[TODO]`

Register `df_customer_summary` as a temporary view and answer four
business questions using pure SQL.

**API hint:**
```python
df.createOrReplaceTempView("view_name")
spark.sql("SELECT ... FROM view_name ...").show()
```

**Questions:**
1. Which country has the highest total revenue?
2. What percentage of total revenue comes from `gold` loyalty-tier customers?
3. What is the average number of orders per loyalty tier?
4. Which single customer placed the most orders?

In [ ]:
# Task 12 [SOLUTION]
df_customer_summary.createOrReplaceTempView('customer_summary')

print('Q1 — Revenue by country:')
spark.sql("""
    SELECT country, ROUND(SUM(total_revenue),2) AS revenue
    FROM customer_summary GROUP BY country ORDER BY revenue DESC""").show()

print('Q2 — Gold tier revenue share:')
spark.sql("""
    SELECT loyalty_tier,
           ROUND(100.0*SUM(total_revenue)/SUM(SUM(total_revenue)) OVER (),2) AS pct
    FROM customer_summary GROUP BY loyalty_tier ORDER BY pct DESC""").show()

print('Q3 — Avg orders per tier:')
spark.sql("""
    SELECT loyalty_tier, ROUND(AVG(total_orders),2) AS avg_orders
    FROM customer_summary GROUP BY loyalty_tier ORDER BY avg_orders DESC""").show()

print('Q4 — Top customer:')
spark.sql("""
    SELECT customer_id,full_name,total_orders
    FROM customer_summary ORDER BY total_orders DESC LIMIT 1""").show()

print('✓ Task 12 passed')